# Label Strategy Comparison

Compare how different label strategies affect learned embedding quality.
Species-level labels (4 classes) provide broader material-class structure;
external_item_id labels (16 classes) provide finer sample-group discrimination.

> **Warning:** Results are diagnostic only. LeaData is a public leather texture
> dataset, not a luxury authentication dataset.

## Run Experiments

From `auth-platform/`:

```bash
# Quick (2 epochs, 100 images)
make label-strategy-comparison-leadata-quick

# Medium (5 epochs, 300 images)
make label-strategy-comparison-leadata-medium

# Full
make label-strategy-comparison-leadata
```

In [ ]:
import json
from pathlib import Path

%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

LS_DIR = Path("../experiments/outputs/label-strategy-comparison")
summary_path = LS_DIR / "label_strategy_comparison_summary.json"
print(f"Exists: {summary_path.exists()}")

In [ ]:
if not summary_path.exists():
    print("No results found. Run: make label-strategy-comparison-leadata-quick")
    data = None
    results = []
else:
    data = json.loads(summary_path.read_text())
    results = [r for r in data.get("results", []) if r.get("evaluation_metrics")]
    print(f"Results: {len(results)}  Failures: {len(data.get('failures', []))}")
    print()

    header = f"{'Loss':<22} {'Labels':<20} {'Classes':>7} {'ClustSep':>10} {'NN Agree':>10} {'R@1':>8}"
    print(header)
    print("-" * len(header))
    for r in results:
        em = r["evaluation_metrics"]
        print(f"{r.get('loss', '?'):<22} {r.get('label_strategy', '?'):<20} {str(r.get('n_classes', '?')):>7} "
              f"{em.get('cluster_separation', 0):>10.4f} "
              f"{em.get('nn_agreement_rate', 0):>9.1%} "
              f"{em.get('recall_at_1', 0):>8.4f}")

In [ ]:
# Cluster separation by loss + label strategy
if results:
    labels_plot = [f"{r.get('loss','?')}\n{r.get('label_strategy','?')}" for r in results]
    seps = [r["evaluation_metrics"].get("cluster_separation", 0) for r in results]

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.barh(labels_plot, seps, color="steelblue")
    ax.set_xlabel("Cluster Separation")
    ax.set_title("Cluster Separation by Loss + Label Strategy")
    for bar, val in zip(bars, seps):
        ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
                f"{val:.4f}", va="center", fontsize=9)
    plt.tight_layout()
    plt.show()

In [ ]:
# Training time comparison
if results:
    times = [r.get("training_time_seconds", 0) for r in results]
    if any(t > 0 for t in times):
        fig, ax = plt.subplots(figsize=(10, 5))
        bars = ax.barh(labels_plot, times, color="seagreen")
        ax.set_xlabel("Training Time (s)")
        ax.set_title("Training Time by Loss + Label Strategy")
        for bar, val in zip(bars, times):
            ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
                    f"{val:.0f}s", va="center", fontsize=9)
        plt.tight_layout()
        plt.show()

## Notes / Conclusions

- **species** (4 classes): broader material-class structure. Measures whether the
  model retrieves same-species leather.
- **external_item_id** (16 classes): finer sample-group labels. Measures whether
  the model retrieves same sample group/item.
- These measure different semantic tasks — direct comparison requires interpretation.
- `texture_baseline_v0.2.0` (0.9418) remains the benchmark for public texture diagnostics.

**These results are diagnostic only.** They do not prove authentication accuracy.